# 05 - Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('Data/cleaned_sold.csv')
df['CloseDate'] = pd.to_datetime(df['CloseDate'])

### Adding New Features/Columns

In [3]:
df.columns

Index(['Flooring', 'ViewYN', 'PoolPrivateYN', 'CloseDate', 'ClosePrice',
       'Latitude', 'Longitude', 'LivingArea', 'MLSAreaMajor', 'CountyOrParish',
       'AttachedGarageYN', 'ParkingTotal', 'SubdivisionName', 'YearBuilt',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'StateOrProvince',
       'FireplaceYN', 'Stories', 'Levels', 'LotSizeArea', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet'],
      dtype='object')

In [ ]:
df['BedBathRatio'] = df['BedroomsTotal'] / np.maximum(df['BathroomsTotalInteger'], 1)
df['AgeProperty'] =  (df['CloseDate'].dt.year - df['YearBuilt']).clip(lower=0)

df['BedBathRatio'] = df['BedBathRatio'].replace([np.inf, -np.inf], np.nan).fillna(0)
df['AgeProperty'] = df['AgeProperty'].replace([np.inf, -np.inf], np.nan).fillna(0)

df = df.drop(columns = ["Flooring"]) #too many nulls and weird formatting, had to remove for better performance/less errors

In [6]:
from utils import create_time_split, get_preprocessing_pipeline

max_date = df['CloseDate'].max()
test_start_date = max_date - pd.DateOffset(months=1)

train_df, test_df = create_time_split(df, 'CloseDate', 12, test_start_date, max_date)

X_train = train_df.drop(columns=['ClosePrice'])
y_train = train_df['ClosePrice']

X_test = test_df.drop(columns=['ClosePrice'])
y_test = test_df['ClosePrice']

preprocessor = get_preprocessing_pipeline(X_train)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

Training Window (X=12 months): 2025-03-30 to 2026-03-30 | Rows: 110241
Testing Window (1 month): 2026-03-30 to 2026-04-30


### Retraining Baseline Linear Model

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

model = LinearRegression()
model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)

print(f"Test Set MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"Test Set R² Score: {r2_score(y_test, y_pred):.4f}")

Test Set MSE: 58336384255.6403
Test Set R² Score: 0.8396


### Retraining Decision Tree Model

In [10]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

decision_tree = DecisionTreeRegressor(random_state=42)

param_grid = {
    'criterion': ['squared_error'],
    'max_depth': [None, 3, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2']
}

grid_search = GridSearchCV(
    estimator=decision_tree,
    param_grid=param_grid,
    cv=3,                            
    scoring='r2', 
    n_jobs=-1,                       
    verbose=1
)

grid_search.fit(X_train_processed, y_train)

print("Best Hyperparameters:", grid_search.best_params_)
print("Best Cross-Validation Score (Negative MSE):", grid_search.best_score_)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_processed)

print("Test Set MSE:", mean_squared_error(y_test, y_pred))
print("Test Set R² Score:", r2_score(y_test, y_pred))

Fitting 3 folds for each of 135 candidates, totalling 405 fits
Best Hyperparameters: {'criterion': 'squared_error', 'max_depth': None, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 10}
Best Cross-Validation Score (Negative MSE): 0.7130157748434942
Test Set MSE: 96089197353.5676
Test Set R² Score: 0.7358485339104986


### Retraining Random Forest Model

In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV

rf_model = RandomForestRegressor()

rf_param_distributions = {
    'n_estimators': [50, 100],
    'criterion': ['squared_error'],
    'max_depth': [5, 10, 20, None],
    'min_samples_split':[2, 5, 10, 15],
    'min_samples_leaf':[1, 2, 4, 8],
    'max_features': ['sqrt', 'log2']
}

rf_random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=rf_param_distributions,
    n_iter=10,                       
    cv=3,                            
    scoring='r2',                     
    n_jobs=-1,                       
    random_state=42,
    verbose=1,
    error_score='raise'               
)

rf_random_search.fit(X_train_processed, y_train)

print("Best RF Hyperparameters:", rf_random_search.best_params_)
print("Best RF Cross-Validation R² Score:", rf_random_search.best_score_)

best_rf_model = rf_random_search.best_estimator_
y_rf_pred = best_rf_model.predict(X_test_processed)

print(f"RF Test Set MSE: {mean_squared_error(y_test, y_rf_pred):.4f}")
print(f"RF Test Set R² Score: {r2_score(y_test, y_rf_pred):.4f}")

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best RF Hyperparameters: {'n_estimators': 50, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None, 'criterion': 'squared_error'}
Best RF Cross-Validation R² Score: 0.8239718570203145
RF Test Set MSE: 61272610632.5805
RF Test Set R² Score: 0.8316


### Comparing Model Performance

In [13]:
compare_table = pd.DataFrame({
    "Model": [
        "Linear Regression", 
        "Decision Tree",
        "Random Forest"
    ],
    "Test R2 Before": [
        0.8408, 
        0.7631,
        0.8065  
    ],
    "Test R2 After": [
        0.8396, 
        0.7358,
        0.8316
    ]
})

compare_table

,Model,Test R2 Before,Test R2 After
0,Linear Regression,0.8408,0.8396
1,Decision Tree,0.7631,0.7358
2,Random Forest,0.8065,0.8316
